# X1 — reduce dbNSFP 5.4a to the SH3 domains

dbNSFP arrived on 27 September 2026, after `DATA_PROVENANCE.md` §9 had recorded it as *requested,
never arrived, not needed*. Nothing was blocked by its absence — ClinVar and gnomAD were taken from
primary sources — so this is an **addition, not a repair**.

### What dbNSFP is for here, and what it is not

dbNSFP carries about **50 precomputed pathogenicity predictors** per missense variant —
AlphaMissense, REVEL, MetaRNN, ESM1b, CADD and the rest. It predicts *whether a variant causes
disease*. Our head predicts **ΔΔG of folding**.

| | |
|---|---|
| **Not** a competitor for **E3** | E3 benchmarks *stability* predictors. Putting a pathogenicity score in that table would be a category error |
| **Is** the external comparison for **E1** | E1 asked whether our stability calls track pathogenicity, and answered yes (rank-biserial +0.615 ≈ AUC 0.81). dbNSFP says what dedicated predictors achieve *on the same variants* |

dbNSFP's own ClinVar benchmark (`dbnsfp_performance_matrix.csv`, shipped with the release) puts the
best predictors at **AUC 0.94–0.99 across all genes**. The interesting question is therefore not
"do we win" — we should not, and we are not trying to — but **whether ΔΔG_fold carries anything
those predictors do not**.

### This notebook acquires and reduces. Nothing else.

It computes no comparison and returns no verdict. That belongs in a program with a pre-registration
written first, as E1, E3 and E4 all had. **Do not add a comparison cell to this notebook.**

Runtime: minutes for sections 1–2, then one streaming pass (hours) for section 3, cached so it is
done once. CPU only.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, gzip, zipfile, struct, subprocess, shutil, time
from pathlib import Path
import pandas as pd, numpy as np

DRIVE = Path('/content/drive/MyDrive')

def pick(*names, what=''):
    for n in names:
        if (DRIVE / n).exists(): return DRIVE / n
    raise FileNotFoundError(f'No {what} folder in MyDrive. Tried: ' + ', '.join(names))

PROJECT = pick('sh3_repro/extended', 'SH3_extended', 'SH3 extended', what='project')
DB      = next((PROJECT / 'input' / n for n in ('dbNSFP', 'external/dbnsfp', 'dbnsfp')
                if (PROJECT / 'input' / n).exists()), PROJECT / 'input' / 'dbNSFP')
OUT     = PROJECT / 'results' / 'X1_dbnsfp_subset'
OUT.mkdir(parents=True, exist_ok=True)
assert DB.exists(), f'dbNSFP folder not found under {PROJECT}/input'
print('project:', PROJECT); print('dbNSFP :', DB)
for f in sorted(DB.iterdir()):
    print(f'  {f.stat().st_size:>15,} B  {f.name}')

## 1. What is actually in these files — probed cheaply, before reading 50 GB

Two files hold the same release in two packagings and together occupy about 100 GB of Drive. This
section reads **only the zip's central directory and the first 64 KB of the gz**, which is fast even
over the FUSE mount, and decides the route before anything expensive starts.

In [ ]:
def probe_gz(path, n=65536):
    """First bytes of a gzip member: is it BGZF, and what is the header line?"""
    with open(path, 'rb') as fh: head = fh.read(n)
    bgzf = head[:2] == b'\x1f\x8b' and b'BC' in head[:20]
    try:
        with gzip.open(path, 'rt', errors='replace') as fh:
            first = fh.readline().rstrip('\n')
    except Exception as e:
        first = f'<could not decompress: {e}>'
    return bgzf, first

INV = []
for f in sorted(DB.iterdir()):
    row = dict(name=f.name, bytes=f.stat().st_size, kind='?', detail='')
    if f.suffix == '.zip':
        with zipfile.ZipFile(f) as z:          # central directory only
            names = z.namelist()
        row.update(kind='zip', detail=f'{len(names)} members')
        ZIP_MEMBERS = names
    elif f.suffix == '.gz':
        bgzf, first = probe_gz(f)
        row.update(kind='bgzf' if bgzf else 'gzip',
                   detail=f'{len(first.split(chr(9)))} tab-columns; starts {first[:60]!r}')
    elif f.suffix == '.csv':
        row.update(kind='csv', detail=f'{sum(1 for _ in open(f)) - 1} rows')
    INV.append(row)
INVENTORY = pd.DataFrame(INV)
print(INVENTORY.to_string(index=False))

In [ ]:
# the zip's members decide whether a per-chromosome route exists
mem = pd.DataFrame({'member': ZIP_MEMBERS})
mem['chrom'] = mem.member.str.extract(r'chr([0-9XYM]+)', expand=False)
print(f'{len(mem)} members, {mem.chrom.notna().sum()} look per-chromosome')
print(mem.head(30).to_string(index=False))
print('\nnon-chromosome members:')
print(mem[mem.chrom.isna()].member.tolist()[:20])

with zipfile.ZipFile(DB / [r['name'] for r in INV if r['kind'] == 'zip'][0]) as z:
    sizes = pd.DataFrame([(i.filename, i.file_size, i.compress_size) for i in z.infolist()],
                         columns=['member', 'uncompressed', 'compressed'])
print(f'\ntotal uncompressed in the zip: {sizes.uncompressed.sum() / 1e9:,.1f} GB')
sizes.to_csv(OUT / 'zip_members_X1.csv', index=False)
INDEXED = [m for m in ZIP_MEMBERS if m.endswith(('.tbi', '.csi'))]
print('tabix/CSI indexes present:', INDEXED or 'NONE - a region query is not possible, '
      'so extraction is one streaming pass')

## 2. The columns we need, located by name rather than by position

dbNSFP's column set changes between releases, so nothing is addressed by index. Every required
column is looked up by name and the cell fails loudly if one is missing.

In [ ]:
HEADER_SRC = None                 # set to a per-chromosome member if the zip has them
def read_header():
    gzs = [r['name'] for r in INV if r['kind'] in ('gzip', 'bgzf') and 'gene' not in r['name']]
    if gzs:
        with gzip.open(DB / gzs[0], 'rt', errors='replace') as fh:
            return gzs[0], fh.readline().rstrip('\n').split('\t')
    z = [r['name'] for r in INV if r['kind'] == 'zip'][0]
    with zipfile.ZipFile(DB / z) as zz:
        m = next(n for n in ZIP_MEMBERS if re.search(r'chr[0-9]+', n) and n.endswith('.gz'))
        with zz.open(m) as raw, gzip.open(raw, 'rt', errors='replace') as fh:
            return m, fh.readline().rstrip('\n').split('\t')

HEADER_SRC, COLS = read_header()
print(f'header from {HEADER_SRC}: {len(COLS)} columns\n')

KEYS = ['#chr', 'pos(1-based)', 'ref', 'alt', 'aaref', 'aaalt', 'aapos',
        'genename', 'Ensembl_proteinid', 'Uniprot_acc', 'HGVSp_VEP']
WANT = ['AlphaMissense_score', 'REVEL_score', 'MetaRNN_score', 'BayesDel_addAF_score',
        'ClinPred_score', 'ESM1b_score', 'CADD_phred', 'CADD_raw', 'PrimateAI_score',
        'gMVP_score', 'VARITY_R_score', 'MutPred_score', 'VEST4_score', 'DEOGEN2_score',
        'Polyphen2_HVAR_score', 'SIFT_score', 'SIFT4G_score', 'PROVEAN_score',
        'MPC_score', 'EVE_score', 'GERP++_RS', 'phyloP100way_vertebrate',
        'clinvar_clnsig', 'clinvar_review', 'clinvar_id',
        'gnomAD4.1_joint_AF', 'gnomAD4.1_joint_AF_popmax']

def find(name):
    if name in COLS: return name
    hits = [c for c in COLS if c.lower().startswith(name.lower().split('_')[0])
            and name.lower().split('_')[-1] in c.lower()]
    return hits[0] if hits else None

missing_keys = [k for k in KEYS if k not in COLS]
assert not missing_keys, f'dbNSFP is missing join columns {missing_keys}; check the release'
RESOLVED = {w: find(w) for w in WANT}
found = {k: v for k, v in RESOLVED.items() if v}
print(f'{len(found)} of {len(WANT)} requested predictor columns resolved')
for k, v in RESOLVED.items():
    print(f'  {"ok  " if v else "MISS"} {k:28s} -> {v or ""}')
KEEP = KEYS + sorted(set(found.values()))
KEEP_IDX = [COLS.index(c) + 1 for c in KEEP]        # 1-based, for awk
json.dump({'header_source': HEADER_SRC, 'n_columns': len(COLS), 'keep': KEEP,
           'resolved': RESOLVED}, open(OUT / 'columns_X1.json', 'w'), indent=1)
print(f'\nkeeping {len(KEEP)} columns')

## 3. The extraction — one streaming pass, filtered to the 235 SH3 genes

No tabix index ships with the release, so a region query is not available and the whole file has to
be read once. `awk` filtering on a hash of the 235 gene names is far faster than pandas here, and
the output is small enough to keep forever.

**The result is cached.** If `dbnsfp_sh3.tsv.gz` already exists the pass is skipped, so this is a
one-off cost even if the session dies.

In [ ]:
GENES = PROJECT / 'reference' / 'sh3_genes_uniprot.tsv'
genes = pd.read_csv(GENES, sep='\t', header=None, names=['gene', 'uniprot'])
print(f'{len(genes)} SH3 genes')
gl = OUT / 'sh3_gene_names.txt'
gl.write_text('\n'.join(sorted(genes.gene.astype(str))) + '\n')

GCOL = COLS.index('genename') + 1
SUBSET = OUT / 'dbnsfp_sh3.tsv.gz'
cut = ','.join(str(i) for i in KEEP_IDX)

AWK = (f"awk -F'\\t' -v OFS='\\t' 'NR==FNR{{g[$1]=1; next}} "
       f'FNR==1{{next}} {{n=split(${GCOL}, a, ";"); keep=0; '
       f'for(i=1;i<=n;i++) if(a[i] in g) keep=1; if(keep) print}}\' '
       f'{gl} -')
print('filter:', AWK[:160], '...')

In [ ]:
def stream_sources():
    """Yield shell commands that write the dbNSFP body (no header) to stdout."""
    zips = [r['name'] for r in INV if r['kind'] == 'zip']
    chrom_members = [m for m in ZIP_MEMBERS if re.search(r'chr[0-9XY]+\.gz$', m)]
    if zips and chrom_members:
        for m in sorted(chrom_members):
            yield m, f'unzip -p {DB / zips[0]!s} {m} | zcat'
        return
    gzs = [r['name'] for r in INV if r['kind'] in ('gzip', 'bgzf') and 'gene' not in r['name']]
    assert gzs, 'no usable variant file found'
    yield gzs[0], f'zcat {DB / gzs[0]!s}'

if SUBSET.exists() and SUBSET.stat().st_size > 0:
    print(f'cached: {SUBSET.name} ({SUBSET.stat().st_size:,} B) - skipping the pass')
else:
    t0 = time.time()
    tmp = Path('/content/dbnsfp_sh3.tsv')
    with tmp.open('w') as fh:
        fh.write('\t'.join(KEEP) + '\n')
    for name, cmd in stream_sources():
        print(f'  {name} ...', end=' ', flush=True)
        full = f'{cmd} | ' + AWK + f' | cut -f{cut} >> {tmp}'
        r = subprocess.run(['bash', '-lc', full])
        print('rc', r.returncode, f'| {tmp.stat().st_size / 1e6:,.1f} MB so far',
              f'| {(time.time() - t0) / 60:.1f} min')
    subprocess.run(['bash', '-lc', f'gzip -c {tmp} > {SUBSET}'], check=True)
    print(f'wrote {SUBSET} ({SUBSET.stat().st_size:,} B) in {(time.time()-t0)/60:.1f} min')

## 4. Restrict to SH3 domain positions, and check it against what we already had

The gene filter keeps whole genes. The atlas supplies the domain boundaries, exactly as E1 did.

The overlap with X0's ClinVar and gnomAD extractions is a **cross-check of our own primary-source
work**: dbNSFP is an independently assembled copy of both, so a large disagreement would mean one of
the two pipelines is wrong.

In [ ]:
# The subset is ~2 GB of text. Loading it whole would need several times that in pandas,
# on a 12.7 GB runtime - so it is streamed, and only the in-domain rows are ever kept.
atlas = pd.read_csv(PROJECT / 'input/frozen/results/SH3_cancer_master.tsv', sep='\t',
                    usecols=['gene', 'uniprot', 'domain_idx', 'dom_start', 'dom_end'])
dom = atlas.drop_duplicates(['gene', 'domain_idx'])
print(f'{len(dom)} SH3 domains define the boundaries')

keep, n_rows, genes_seen = [], 0, set()
t0 = time.time()
# dbNSFP writes '.' for a missing value, not an empty field. Without na_values every
# predictor column looks 100% covered, which is how the first run reported it.
for ch in pd.read_csv(SUBSET, sep='\t', chunksize=400_000, low_memory=False,
                      na_values=['.', './.', '']):
    n_rows += len(ch)
    ch['gene1'] = ch.genename.astype(str).str.split(';').str[0]
    ch['aapos_i'] = pd.to_numeric(ch.aapos.astype(str).str.split(';').str[0], errors='coerce')
    genes_seen.update(ch.gene1.unique())
    jj = ch.merge(dom, left_on='gene1', right_on='gene')
    jj = jj[(jj.aapos_i >= jj.dom_start) & (jj.aapos_i <= jj.dom_end)]
    if len(jj): keep.append(jj)
    if n_rows % 4_000_000 == 0:
        print(f'  {n_rows:,} rows read, {sum(map(len, keep)):,} in-domain '
              f'({(time.time()-t0)/60:.1f} min)', flush=True)

j = pd.concat(keep, ignore_index=True) if keep else pd.DataFrame()
j['pos_in_domain'] = j.aapos_i - j.dom_start + 1
j.to_parquet(OUT / 'dbnsfp_sh3_domains.parquet', index=False)
SH3SET = set(genes.gene.astype(str))
n_genes = len(genes_seen & SH3SET)          # SH3 genes actually matched
n_first = len(genes_seen)                   # distinct FIRST-listed genes, SH3 or not
print(f'\n{n_rows:,} dbNSFP rows')
print(f'  {n_genes} of {len(genes)} SH3 genes matched')
print(f'  {n_first} distinct first-listed gene names among the kept rows - larger than',
      '235 because dbNSFP lists every overlapping gene in one semicolon-separated field,')
print('  and a row is kept when ANY of them is an SH3 gene. Not a filter failure.')
n_dom = j.groupby(['gene', 'domain_idx']).ngroups
print(f'{len(j):,} rows inside an SH3 domain, {j.gene.nunique()} genes, {n_dom} domains')

cov = {c: float(j[c].notna().mean()) for c in KEEP if c in j.columns and c not in KEYS}
COV = pd.Series(cov).sort_values(ascending=False)
print('\npredictor coverage inside SH3 domains:')
print((COV * 100).round(1).to_string())

In [ ]:
# cross-check against X0's own extractions
chk = {}
XOUT = PROJECT / 'results' / 'X0_external_data'
if (XOUT / 'clinvar_sh3.csv').exists():
    cv = pd.read_csv(XOUT / 'clinvar_sh3.csv')
    cv['pos_protein'] = pd.to_numeric(cv.pos_protein, errors='coerce')
    # compare like with like: OUR ClinVar spans whole genes, dbNSFP here is domains only,
    # so restrict ours to the same domain windows before counting the overlap
    cvd = cv.merge(dom, left_on='GeneSymbol', right_on='gene')
    cvd = cvd[(cvd.pos_protein >= cvd.dom_start) & (cvd.pos_protein <= cvd.dom_end)]
    ours_all = set(zip(cv.GeneSymbol, cv.pos_protein, cv.wt_aa, cv.mut_aa))
    ours = set(zip(cvd.GeneSymbol, cvd.pos_protein, cvd.wt_aa, cvd.mut_aa))
    theirs = set(zip(j.gene1, j.aapos_i, j.aaref, j.aaalt))
    pct = round(100 * len(ours & theirs) / max(len(ours), 1), 1)
    chk = dict(clinvar_keys_ours_all_genes=len(ours_all),
               clinvar_keys_ours_in_domain=len(ours), dbnsfp_keys_in_domain=len(theirs),
               shared=len(ours & theirs), recovered_pct=pct)
    print(f'ClinVar in SH3 domains: ours {len(ours):,} | dbNSFP {len(theirs):,} | '
          f'shared {len(ours & theirs):,} ({pct}% of ours)')
    print(f'  (our whole-gene ClinVar set is {len(ours_all):,}; most of it lies outside the domains)')
if 'clinvar_clnsig' in j.columns:
    print('\ndbNSFP clinvar_clnsig inside SH3 domains:')
    print(j.clinvar_clnsig.value_counts().head(8).to_string())
json.dump(chk, open(OUT / 'crosscheck_X1.json', 'w'), indent=1)

## 5. The gene-level file, and dbNSFP's own predictor benchmark

`dbNSFP5.4_gene.gz` is 23 MB and carries per-gene constraint and annotation. The performance matrix
is dbNSFP's **own** ClinVar benchmark of every score it ships — it is the published reference any
later comparison should be read against, so it is recorded here rather than re-derived.

In [ ]:
gz = [f for f in DB.iterdir() if 'gene' in f.name and f.suffix == '.gz']
if gz:
    G = pd.read_csv(gz[0], sep='\t', low_memory=False)
    Gs = G[G.iloc[:, 0].astype(str).isin(set(genes.gene.astype(str)))]
    Gs.to_csv(OUT / 'dbnsfp_gene_sh3.tsv', sep='\t', index=False)
    print(f'gene-level: {len(G):,} genes, {len(Gs)} of ours kept, {G.shape[1]} columns')

pm = [f for f in DB.iterdir() if 'performance' in f.name]
if pm:
    P = pd.read_csv(pm[0])
    cur = P[(P.ClinVar_Year == P.ClinVar_Year.max()) &
            P.Stratum.str.contains('Star 1 Single [+]')].sort_values('AUC', ascending=False)
    cur.to_csv(OUT / 'dbnsfp_published_auc.csv', index=False)
    print(f'\ndbNSFP published AUC, ClinVar {P.ClinVar_Year.max()}, star 1+ missense (top 12):')
    print(cur[['Score', 'AUC', 'Coverage', 'N_Path', 'N_Benign']].head(12).to_string(index=False))

## 6. RESULTS_X1.md

In [ ]:
L = ['# X1 - dbNSFP 5.4a reduced to the SH3 domains', '',
     f'Run {pd.Timestamp.now():%Y-%m-%d %H:%M}. Machine-written. **Acquisition only** - this '
     'program computes no comparison and returns no verdict.', '',
     '## What was read', '',
     '| file | bytes | kind | detail |', '|---|---|---|---|']
L += [f'| `{r.name}` | {r.bytes:,} | {r.kind} | {r.detail} |' for r in INVENTORY.itertuples()]
L += ['', f'No tabix or CSI index ships with the release, so the subset was built by one '
      'streaming pass filtered on the 235 SH3 gene names.', '',
      '## What came out', '',
      f'- `dbnsfp_sh3.tsv.gz` - {n_rows:,} rows, {n_genes} genes, {len(KEEP)} columns',
      f'- `dbnsfp_sh3_domains.parquet` - **{len(j):,} rows inside an SH3 domain**, '
      f'{j.gene.nunique()} genes, {j.groupby(["gene","domain_idx"]).ngroups} domains',
      f'- `dbnsfp_gene_sh3.tsv`, `dbnsfp_published_auc.csv`, `columns_X1.json`', '',
      '## Predictor coverage inside SH3 domains', '',
      '| score | % of in-domain rows with a value |', '|---|---|']
L += [f'| {k} | {v*100:.1f} % |' for k, v in COV.items()]
L += ['', '## What this is for', '',
      'dbNSFP predicts **pathogenicity**; our head predicts **ΔΔG of folding**. It is therefore '
      'not a competitor for E3, which benchmarks stability predictors. It is the external '
      'comparison for E1, whose question was whether our stability calls track pathogenicity.',
      '', 'Any such comparison needs its own pre-registration, written before the numbers exist.']
(OUT / 'RESULTS_X1.md').write_text('\n'.join(L))
print('\n'.join(L))
print('\nwritten to', OUT)
for p in sorted(OUT.iterdir()):
    if p.is_file(): print(f'  {p.stat().st_size:>13,} B  {p.name}')